# Robinhood — fetch

Robinhood-specific data fetching, isolated here. The **generic** rebuild / reconcile /
curate steps live in `refresh.ipynb`.

**Token:** the bearer JWT lasts ~weeks. When expired, grab a fresh
`authorization: Bearer …` header from any `api.robinhood.com` request in Chrome
DevTools and paste it into `data/raw/robinhood/.rh_token`. The cell below shows how
much life the current token has.

In [ ]:
import json, base64
import pandas as pd
from invest.broker import robinhood

tok_path = robinhood.TOKEN_PATH
if tok_path.exists():
    payload = tok_path.read_text().strip().split(".")[1]
    payload += "=" * (-len(payload) % 4)
    exp = pd.Timestamp(json.loads(base64.urlsafe_b64decode(payload))["exp"], unit="s")
    left = (exp - pd.Timestamp.now()).days
    print(f"RH token expires {exp:%Y-%m-%d} — {left} days left " + ("✅" if left > 0 else "⚠️ EXPIRED, refresh it"))
else:
    print("No .rh_token yet — paste a fresh Bearer token into data/raw/robinhood/.rh_token")

## Fetch (history + positions)

Incremental: merges history records by id and overwrites `positions.json` — a no-op
when nothing changed. Then open **`refresh.ipynb`** to rebuild + reconcile.

In [ ]:
import sys
!{sys.executable} -m invest.broker.robinhood.fetch